# Flag Algebras in Practice: A Zászló Tutorial

**Zászló** is a Python library implementing the *flag algebra method* for extremal combinatorics.
Starting from a problem specification — which graphs are forbidden, what density to bound? — it
enumerates combinatorial structures, builds a semidefinite program (SDP), solves it, and
verifies the certificate in exact rational arithmetic.

This tutorial works through four problems:

1. **Mantel's theorem** — maximum edge density in triangle-free graphs is $\frac{1}{2}$
2. **Rational certificates** — rounding a floating-point SDP solution into a formal proof
3. **The pentagon problem** — maximum $C_5$ density in triangle-free graphs is $\frac{24}{625}$
4. **$K_4^-$-free 3-graphs** — maximum edge density in $K_4^-$-free 3-uniform hypergraphs is $\frac{1}{3}$

## Getting Set Up

Run the cell below to load everything we need. Click it and press **Shift+Enter** — you'll do the same for every code cell throughout.

In [ ]:
from fractions import Fraction

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from IPython.display import display

from zaszlo import (
    FlagProblem,
    Hypergraph,
    build_flag_algebra_data,
    certify,
    identify_sharps,
    k4_minus,
    round_certificate,
    solve_sdp,
    verify_certificate,
)

plt.rcParams.update({"figure.dpi": 100})

The two helpers below draw grids of graphs and hypergraphs. They're used throughout for visualization — gold nodes (marked ★) indicate sharp, i.e. extremal, graphs.

In [ ]:
def draw_k2_graphs(graphs, densities=None, sharps=None, ncols=4, suptitle=None):
    n = len(graphs)
    if n == 0:
        return
    ncols = min(ncols, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 2.6 * nrows))
    flat = np.array(axes).reshape(-1)
    for i, g in enumerate(graphs):
        ax = flat[i]
        G = nx.Graph()
        G.add_nodes_from(range(1, g.n + 1))
        G.add_edges_from(g.edges)
        is_sharp = sharps is not None and i in sharps
        nx.draw(
            G, nx.circular_layout(G), ax=ax, with_labels=True,
            node_color="#F5A623" if is_sharp else "#4A90D9",
            node_size=420, font_size=9, font_color="white",
            edge_color="#444", width=2,
        )
        label = f"d = {densities[i]}" if densities is not None else f"H{i}"
        if is_sharp:
            label += "  ★"
        ax.set_title(label, fontsize=8, pad=3,
                     color="#C05000" if is_sharp else "#333")
    for j in range(n, len(flat)):
        flat[j].set_visible(False)
    if suptitle:
        fig.suptitle(suptitle, fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()


def draw_k3_hypergraphs(graphs, densities=None, sharps=None, ncols=4, suptitle=None):
    n = len(graphs)
    if n == 0:
        return
    ncols = min(ncols, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 2.6 * nrows))
    flat = np.array(axes).reshape(-1)
    for i, g in enumerate(graphs):
        ax = flat[i]
        is_sharp = sharps is not None and i in sharps
        face = "#F5A623" if is_sharp else "#4A90D9"
        n_v = g.n
        angles = [2 * np.pi * v / n_v - np.pi / 2 for v in range(n_v)]
        pos = {v + 1: (np.cos(angles[v]), np.sin(angles[v])) for v in range(n_v)}
        for edge in g.edges:
            tri = plt.Polygon(
                [pos[v] for v in edge], alpha=0.4,
                facecolor=face, edgecolor="#333", lw=1.2,
            )
            ax.add_patch(tri)
        for v, (x, y) in pos.items():
            ax.plot(x, y, "o", color="#111", ms=7, zorder=3)
            ax.text(x * 1.38, y * 1.38, str(v),
                    ha="center", va="center", fontsize=9, zorder=4)
        ax.set(xlim=(-1.7, 1.7), ylim=(-1.7, 1.7), aspect="equal")
        ax.set_axis_off()
        label = f"d = {densities[i]}" if densities is not None else f"H{i}"
        if is_sharp:
            label += "  ★"
        ax.set_title(label, fontsize=8, pad=3,
                     color="#C05000" if is_sharp else "#333")
    for j in range(n, len(flat)):
        flat[j].set_visible(False)
    if suptitle:
        fig.suptitle(suptitle, fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## Background: The Flag Algebra Method

The flag algebra method (Razborov, 2007) turns extremal combinatorics questions into semidefinite
programs. Here is the 60-second version.

**The setting.** We want to bound the density of a target pattern $H$ in large graphs that avoid a
forbidden subgraph $F$. The *density* $p(H; G)$ is the probability that a uniformly random
$|V(H)|$-subset of $V(G)$ induces a copy of $H$.

**The averaging identity.** A *type* $\sigma$ is a small fully-labeled $k$-graph. A *flag* over
$\sigma$ is a larger graph whose first $s$ vertices replicate $\sigma$ exactly; the remaining
vertices are *unlabeled* and averaged over all embeddings. For any large admissible graph $G$,
the probability of a random extension landing on flag pair $(f_i, f_j)$ is the *pair density*
$p(f_i, f_j; G)$ — a linear function of the subgraph densities of $G$.

**The SDP.** For any positive semidefinite matrices $\{Q_\sigma\}$, the quantity
$$\sum_{\sigma}\sum_{i,j} (Q_\sigma)_{ij}\, p(f_i^\sigma, f_j^\sigma; G) \;\ge\; 0$$
is non-negative. So if we find $\{Q_\sigma\}$ and $\lambda$ with
$$\lambda - p(H; G) = \sum_\sigma \langle Q_\sigma, P_\sigma(H) \rangle \ge 0$$
for every admissible $H$, then $p(H; G) \le \lambda$ for all admissible $G$.
Minimizing $\lambda$ subject to these constraints is a semidefinite program.

**The Zászló workflow:**
```
data   = build_flag_algebra_data(prob)     # enumerate structures, compute pair densities
result = solve_sdp(data, extract_Q=True)  # find optimal PSD certificate
cert   = verify_certificate(data, result)  # verify in exact rational arithmetic
sharps = identify_sharps(data, result)    # find extremal (tight) examples
```

---
## 1. Mantel's Theorem

**Theorem (Mantel, 1907).** A triangle-free graph on $n$ vertices has at most
$\lfloor n^2/4 \rfloor$ edges. Equivalently, its edge density satisfies
$p(K_2; G) \le \frac{1}{2}$, with equality for the balanced complete bipartite graph
$K_{\lfloor n/2\rfloor,\lceil n/2\rceil}$.

We'll recover the bound $\frac{1}{2}$ via flag algebras at order $n = 4$.

### Defining the problem

First, the forbidden graph: $K_3$, the triangle. We construct it by supplying the vertex count,
edge uniformity ($k = 2$ for ordinary graphs), and the edge list. Vertices in Zászló are 1-based.

In [ ]:
K3 = Hypergraph(3, 2, [(1, 2), (1, 3), (2, 3)])
K3

Every Zászló object renders as an annotated HTML card in Jupyter — just put it on the last line
of a cell. For a richer plain-text description that works anywhere (scripts, REPLs, notebooks),
call `.explain()`. We'll use both throughout.

Now let's encode the optimization problem. `FlagProblem` bundles all the parameters the library
needs to build and solve the SDP.

In [ ]:
prob_mantel = FlagProblem(
    4,              # n: admissible graphs on 4 vertices
    2,              # type_order: types of order 0 and 2 (same parity as n=4)
    2,              # k: ordinary graphs (2-uniform)
    forbidden=[K3], # forbid non-induced K3
    minimize=False, # upper bound mode: minimize lambda
)
prob_mantel

### Parameters explained

| Parameter | Value | Meaning |
|---|---|---|
| `n` | 4 | Admissible graphs (the "test objects") live on 4 vertices |
| `type_order` | 2 | Use types of order 0 and 2; must share parity with $n$ |
| `k` | 2 | Ordinary graphs (edges have size 2) |
| `forbidden` | `[K3]` | No non-induced copy of $K_3$ is allowed |
| `minimize` | `False` | Upper bound mode: minimize $\lambda$ |

For a type of order $s$, flags live on $m = \frac{n+s}{2}$ vertices. With $n = 4$:

- Order-$0$ type → flags on $m = 2$ vertices
- Order-$2$ types → flags on $m = 3$ vertices

Larger $n$ and `type_order` give tighter bounds at the cost of a bigger SDP.
`type_order = n - 2` is the standard choice.

### Building the flag algebra data

`build_flag_algebra_data` enumerates all non-isomorphic types, flags, and admissible graphs
consistent with the problem, then computes the pair density matrices used as SDP constraints.

In [ ]:
data_mantel = build_flag_algebra_data(prob_mantel)
data_mantel

In [ ]:
print(data_mantel.explain())

### Admissible graphs

All 7 non-isomorphic triangle-free graphs on 4 vertices. These are the objects the flag algebra
"sees" — any triangle-free graphon decomposes into a mixture of their densities.

In [ ]:
draw_k2_graphs(
    data_mantel.admissible,
    densities=data_mantel.densities,
    suptitle="All 7 triangle-free graphs on 4 vertices",
)

### Types and flags

A **type** of order $s$ is a fully-labeled $k$-graph on $s$ vertices. In Zászló's HTML rendering,
labeled (fixed) vertices are shown in **orange** and unlabeled (averaged) vertices in **blue**.

A **flag** over type $\sigma$ is a graph on $m$ vertices whose first $s$ vertices replicate
$\sigma$ exactly; the remaining $m - s$ unlabeled vertices are averaged over all embeddings into a
large host graph $G$. That averaging turns flag products into linear constraints on subgraph densities.

In [ ]:
# Type 2: the single edge {1,2} -- both vertices labeled (orange)
display(data_mantel.types[2])

# A flag over type 2: vertex 3 is unlabeled (blue) and gets averaged
f = data_mantel.flags[2][1]
display(f)
print(f.explain())

### Solving the SDP

`solve_sdp` uses Clarabel to find the optimal $\lambda$ and PSD matrices $\{Q_\sigma\}$.
Pass `extract_Q=True` to keep the certificate matrices for later verification.

In [ ]:
result_mantel = solve_sdp(data_mantel, extract_Q=True)
print(result_mantel.explain(data=data_mantel))

### Verifying the certificate

`verify_certificate` converts the SDP solution to exact rational arithmetic:

1. **Rationalize** $\lambda$ and each $Q_\sigma$ entry-wise (float → `Fraction` via `limit_denominator`)
2. **Check PSD**: each $Q_\sigma$ must have no significant negative eigenvalue
3. **Compute residuals**: for each admissible $H$, compute
   $\lambda - \mathrm{density}(H) - \sum_\sigma \langle Q_\sigma, P_\sigma(H)\rangle$ exactly

`valid: True` requires every residual to be $\ge 0$. Direct rationalization of a floating-point
$Q$ can introduce tiny constraint violations, so `valid: False` with `min_residual` near zero is
a common outcome — the bound $\lambda$ is still accurate. Section 2 shows how to produce a
provably valid certificate.

In [ ]:
cert_mantel = verify_certificate(data_mantel, result_mantel)
print(f"Bound (exact)  : {cert_mantel['lam_certified']}")
print(f"Min PSD eigval : {cert_mantel['min_psd_eigval']:.2e}")
print(f"Min residual   : {float(cert_mantel['min_residual']):.2e}")
print(f"Valid          : {cert_mantel['valid']}")

### Sharp (extremal) graphs

An admissible graph $H$ is *sharp* when its SDP slack is zero: the bound is tight at $H$.
Sharp graphs reveal the extremal structure — for Mantel, the extremal graphon is the balanced
bipartite graphon ($K_{n/2,n/2}$ in the limit), so locally one sees $C_4 = K_{2,2}$ and all
its subgraphs.

In [ ]:
sharps_mantel = identify_sharps(data_mantel, result_mantel)
sharps_mantel

In [ ]:
draw_k2_graphs(
    data_mantel.admissible,
    densities=data_mantel.densities,
    sharps=set(sharps_mantel.indices),
    suptitle="Triangle-free graphs on 4 vertices  (gold ★ = sharp/extremal)",
)

---
## 2. Rational Certificates via `round_certificate`

The SDP solver returns floating-point $Q$ matrices. When we call `verify_certificate` on these
directly, the residuals may be slightly negative (at the level of solver tolerance), so `valid`
may be `False` even when the bound is numerically correct. This is expected, not a bug.

To produce a certificate that constitutes a **genuine mathematical proof**, we need exact rational
$Q_\sigma$ matrices that are provably PSD and satisfy all constraints. `round_certificate`
does this in three steps for each $Q_\sigma$:

1. **Cholesky factor**: decompose $Q_\sigma \approx L L^\top$ where $L$ is lower triangular.
2. **Round $L$**: replace each entry with the nearest rational having denominator $\le D$
   (controlled by `denom_limit`, default $D = 1000$), in exact `Fraction` arithmetic.
3. **Reconstruct**: form $\tilde{Q}_\sigma = \tilde{L}\,\tilde{L}^\top$ exactly — so
   $\tilde{Q}_\sigma \succeq 0$ by construction.

Because rounding changes the flag sums, `round_certificate` recomputes the certified bound
$\lambda_{\rm cert}$ as the tightest $\lambda$ the rounded $Q$ matrices actually prove.
It's slightly weaker than the solver's bound, but honest and independently checkable.

### Step 1 — the raw certificate

We already computed `result_mantel` above. Verifying it directly shows the `valid: False` baseline.

In [ ]:
cert_raw = verify_certificate(data_mantel, result_mantel)
print(f"valid          : {cert_raw['valid']}")
print(f"min_residual   : {float(cert_raw['min_residual']):.2e}  <- tiny float noise")
print(f"min_psd_eigval : {cert_raw['min_psd_eigval']:.2e}")
print(f"lam_certified  : {cert_raw['lam_certified']}")

### Step 2 — round to exact rationals

`denom_limit=1000` means each Cholesky factor entry is rounded to the nearest rational with
denominator at most 1000. Larger values give a tighter certified bound at the cost of bigger numerators.

In [ ]:
result_mantel_rat = round_certificate(data_mantel, result_mantel, denom_limit=1000)

print(f"Original bound : {result_mantel.bound:.10f}  (float)")
print(f"Certified bound: {result_mantel_rat.bound_exact}")
print(f"               = {float(result_mantel_rat.bound_exact):.10f}  (slightly above 1/2; closes with larger denom_limit)")

### Step 3 — verify the rounded certificate

With $\tilde Q_\sigma = \tilde L \tilde L^\top$ exact in `Fraction` arithmetic, every
residual is now non-negative exactly, and `valid` is `True`.

In [ ]:
cert_rat = verify_certificate(data_mantel, result_mantel_rat)
print(f"valid          : {cert_rat['valid']}")
print(f"lam_certified  : {cert_rat['lam_certified']}")
print(f"min_residual   : {cert_rat['min_residual']}  (exact zero)")
print()
print("Residuals per admissible graph:")
for i, (H, r) in enumerate(zip(data_mantel.admissible, cert_rat["residuals"])):
    status = "\u2713" if r >= 0 else "\u2717"
    print(f"  H{i}  ({len(H.edges)} edges)  residual = {r}  {status}")

In [ ]:
proof_mantel = certify(result_mantel)
print(proof_mantel.explain())

### The full proof pipeline

`certify()` wraps `round_certificate` and `verify_certificate` into a single call, returning a `Certificate` object with exact rational $Q_\sigma$ matrices, a certified bound, per-graph residuals, and a validity flag.

```python
data   = build_flag_algebra_data(prob)
result = solve_sdp(data, extract_Q=True)
proof  = certify(result)    # round + verify; Certificate with exact Fraction arithmetic
assert proof.valid          # proof complete
sharps = identify_sharps(data, result)  # find sharp (tight-constraint) graphs
```

**Two paths, two guarantees.**

| Path | Bound | Residuals | Proof? |
|------|-------|-----------|--------|
| Raw float output | solver bound (e.g. $1/2$) | may be $\sim{-10^{-9}}$ (float noise) | No |
| `certify(result)` | $\lambda_{\rm cert}$ (slightly weaker) | all $\ge 0$ exactly | Yes |

**Note on `denom_limit`.** `certify` accepts a `denom_limit` keyword (default 1000) that controls rounding precision. Larger values give a tighter certified bound at the cost of bigger numerators.

---
## 3. The Pentagon Problem

**Problem.** Among all triangle-free graphs, what is the maximum density of 5-cycles ($C_5$)?

**Answer** (Grzesik, 2012; Hatami–Hladký–Král’–Norine–Razborov, 2013):
$\dfrac{24}{625} = 0.0384$.

The extremal construction is the *balanced blow-up of $C_5$*: replace each vertex of $C_5$ by an
equal-size independent set and connect consecutive parts completely. In the limit this achieves
exactly $\frac{24}{625}$.

The new ingredient here is the `target` parameter, which tells Zászló to optimize the density of
a specific pattern rather than edge density. We also raise $n$ to 5 and `type_order` to 3 to get
the tight bound.

**Note on induced vs. non-induced.** In a triangle-free graph every $C_5$ is automatically
induced — any chord would close a triangle — so the $C_5$ density and induced $C_5$ density
coincide here, and `target=C5` (which computes induced density) is correct without qualification.

In [ ]:
C5 = Hypergraph(5, 2, [(1, 2), (2, 3), (3, 4), (4, 5), (5, 1)])
C5

In [ ]:
prob_c5 = FlagProblem(
    5,              # n=5: work at order 5
    3,              # type_order=3 (same parity as 5; flags on (5+3)/2=4 vertices)
    2,              # k=2: ordinary graphs
    forbidden=[K3], # still triangle-free
    target=C5,      # maximize C5 density, not edge density
    minimize=False,
)
prob_c5

In [ ]:
data_c5 = build_flag_algebra_data(prob_c5)
print(data_c5.explain())

With `target=C5`, the **density of each admissible graph** is now its induced $C_5$ density. The
14 admissible triangle-free graphs on 5 vertices are shown below — most have zero $C_5$ density,
with the full 5-cycle being the only one that achieves $d = 1$.

In [ ]:
c5_dens_float = [round(float(d), 4) for d in data_c5.densities]
draw_k2_graphs(
    data_c5.admissible,
    densities=c5_dens_float,
    ncols=5,
    suptitle="Triangle-free graphs on 5 vertices  (induced C₅ density shown)",
)

In [ ]:
result_c5 = solve_sdp(data_c5, extract_Q=True)
cert_c5 = certify(result_c5)
print(result_c5.explain(data=data_c5))
print(f"Known exact: {Fraction(24, 625)}  =  {24/625:.6f}")

In [ ]:
sharps_c5 = identify_sharps(data_c5, result_c5)
sharps_c5

In [ ]:
draw_k2_graphs(
    data_c5.admissible,
    densities=c5_dens_float,
    sharps=set(sharps_c5.indices),
    ncols=5,
    suptitle="Triangle-free graphs on 5 vertices  (gold ★ = sharp/extremal)",
)

---
## 4. $K_4^-$-Free 3-Uniform Hypergraphs

**Generalization to hypergraphs.** In a *$k$-uniform hypergraph* every edge has exactly $k$
vertices. Flag algebras apply unchanged — just set `k=3`.

**The forbidden pattern.** $K_4^-$ is the unique (up to isomorphism) 3-uniform hypergraph on
4 vertices with exactly 3 edges (one edge is missing from the complete $K_4^{(3)}$):
$$K_4^- = \bigl\{\{1,2,3\},\, \{1,2,4\},\, \{1,3,4\}\bigr\}$$

**The question.** What is the maximum edge density of a $K_4^-$-free 3-uniform hypergraph?

**Answer** (Frankl–Füredi conjecture; Keevash–Sudakov, 2005): $\dfrac{1}{3}$.
The extremal construction partitions the vertex set into 3 equal parts and includes all triples
that have at least two vertices in one part.

The built-in `k4_minus()` shorthand constructs this graph.

In [ ]:
K4m = k4_minus()
K4m

In [ ]:
print(K4m.explain())

In [ ]:
prob_k4m = FlagProblem(
    5,               # n=5
    3,               # type_order=3 (same parity as 5)
    3,               # k=3: 3-uniform hypergraphs
    forbidden=[K4m],
    minimize=False,
)
prob_k4m

In [ ]:
data_k4m = build_flag_algebra_data(prob_k4m)
print(data_k4m.explain())

### Visualizing 3-uniform hypergraphs

Each hyperedge (a triple of vertices) is drawn as a filled triangle. The 11 admissible graphs
below are all $K_4^-$-free 3-uniform hypergraphs on 5 vertices.

In [ ]:
draw_k3_hypergraphs(
    data_k4m.admissible,
    densities=data_k4m.densities,
    ncols=4,
    suptitle="All 11 K₄⁻-free 3-uniform hypergraphs on 5 vertices",
)

In [ ]:
result_k4m = solve_sdp(data_k4m, extract_Q=True)
print(result_k4m.explain(data=data_k4m))

In [ ]:
cert_k4m = certify(result_k4m)
print(f"Bound (exact)  : {cert_k4m.bound}")
print(f"Valid          : {cert_k4m.valid}")

In [ ]:
sharps_k4m = identify_sharps(data_k4m, result_k4m)
sharps_k4m

In [ ]:
draw_k3_hypergraphs(
    data_k4m.admissible,
    densities=data_k4m.densities,
    sharps=set(sharps_k4m.indices),
    ncols=4,
    suptitle="K₄⁻-free 3-graphs on 5 vertices  (gold ★ = sharp/extremal)",
)

---
## Summary

| Example | `k` | `n` | Forbidden | Target | Bound | Reference |
|---|---|---|---|---|---|---|
| Mantel's theorem | 2 | 4 | $K_3$ | edge density | $\frac{1}{2}$ | Mantel (1907) |
| Pentagon density | 2 | 5 | $K_3$ | $C_5$ density | $\frac{24}{625}$ | Grzesik (2012); Hatami et al. (2013) |
| $K_4^-$-free 3-graphs | 3 | 5 | $K_4^-$ | edge density | $\frac{1}{3}$ | Frankl–Füredi / Keevash–Sudakov |

### The proof pipeline

```python
data   = build_flag_algebra_data(prob)          # enumerate structures, compute pair densities
result = solve_sdp(data, extract_Q=True)        # numerical SDP
proof  = certify(result)                        # round + verify; Certificate in exact arithmetic
assert proof.valid                              # proof complete
sharps = identify_sharps(data, result)          # find sharp (tight-constraint) graphs
```

### `FlagProblem` quick reference

```python
FlagProblem(
    n,                       # order of admissible graphs
    type_order,              # max type order; same parity as n, and <= n-2
    k,                       # edge uniformity (k=2: graphs, k=3: 3-uniform, ...)
    forbidden=[...],         # forbid non-induced copies
    forbidden_induced=[...], # forbid induced copies (optional)
    target=H,                # optimize density of H (default: edge density)
    minimize=False,          # False = upper bound, True = lower bound
)
```

### Built-in hypergraph constructors

```python
Hypergraph(n, k, edges)     # general constructor; vertices are 1-based
complete(n, k=2)            # complete k-uniform hypergraph on n vertices
k4_minus()                  # K4-minus: 4-vertex 3-graph with 3 edges
c5_3uniform()               # tight 5-cycle in 3-uniform hypergraphs
f32()                       # F32: 5-vertex 3-graph with 4 edges
```

In [ ]:
rows = [
    ("Mantel",     proof_mantel.bound, Fraction(1, 2)),
    ("C5 density", cert_c5.bound,      Fraction(24, 625)),
    ("K4- free",   cert_k4m.bound,     Fraction(1, 3)),
]
print(f"{'Problem':<14}  {'SDP bound':>18}  {'Known exact':>12}  {'|error|':>10}")
print("-" * 62)
for name, lam_rat, lam_exact in rows:
    err = abs(float(lam_rat) - float(lam_exact))
    print(f"{name:<14}  {float(lam_rat):>18.6f}  {str(lam_exact):>12}  {err:>10.2e}")